# 模块一：SwellEx-96 纯 CPU 原始数据预处理与 Google Drive 持久化

本 Notebook 专为 **Google Colab 纯 CPU 运行时**（`Runtime -> Change runtime type -> CPU`）设计，完全不调用 PyTorch / CUDA，避免消耗宝贵的 Colab GPU 计算额度。

### 工作流职责与 Google Drive 目录约定
1. **自动定位原始数据（只读）**：默认绑定你的 Google Drive 原始数据文件夹 [`1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj`](https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link)：
   - 同时兼容 **直接把 `Origindata` 内部四项（`environments/`, `events/`, `positions/`, `source/`）拷入该文件夹**，或 **在该文件夹下保留 `Origindata/` 子目录** 两种结构；
   - 优先通过 Colab 挂载盘（`user.drive.id` 扩展属性 / 快捷方式 ID / 目录特征）零额外授权定位，若未出现在 `MyDrive` 则自动回退至 **Google Drive API v3** 按 Folder ID 分页拉取（规避 `gdown` 在 `ctds/` 52 文件目录上的 50 文件截断限制）；
   - 将特征计算所需的 24 个原始文件（约 367 MB）一次性缓存到实例本地高速 SSD（`/content/swellex_work/Origindata`），避免 `base_pack` 多轮读取和 SHA-256 校验反复穿透 Google Drive FUSE。
2. **本地高速盘特征提取**：所有 `scipy` / `numpy` 频域特征计算与 `h5py` 写入均在本地 SSD（`/content/swellex_work/outputs/`）完成，彻底规避 Drive FUSE 不支持软链接及随机写入慢的问题。
3. **落盘至 Google Drive 并生成索引清单**：将生成的各实验包 HDF5 数据集同步到 `/content/drive/MyDrive/SwellEx_Colab/datasets/`（保持原始数据目录只读不污染），写入 `dataset_manifest.json` 供 **模块二（`02_network_training_gpu.ipynb`）** 直接消费。

> **注意**：在 Colab 内请直接按顺序运行下方单元格，**不要**调用 `./scripts/run.sh`（该脚本用于本地通过 SSH 派发到 `haiqin` 服务器）。

In [ ]:
# Cell 1: 环境检测、仓库引导、Google Drive 挂载与 Origindata (1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj) 自动适配
from collections import deque
from pathlib import Path
import importlib.util
import io
import os
import re
import shutil
import subprocess
import sys

# --- Google Drive 与仓库配置（已绑定你的 Origindata 文件夹链接） ---
ORIGINDATA_DRIVE_URL = "https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link"
ORIGINDATA_LOCAL_OVERRIDE = os.environ.get("SWELLEX_ORIGINDATA_DIR", "").strip()  # 若需手动指定本地/挂载路径可填写
DRIVE_WORKSPACE_SUBDIR = "SwellEx_Colab"  # 派生 HDF5 与实验输出保存在 MyDrive/<此目录>/ 下，不写入原始数据目录
REPO_GIT_URL = "https://github.com/AmazingUmi/SwellEx.git"
REPO_GIT_BRANCH = "main"
AUTO_UPDATE_REPO = True     # Colab 下每次运行 Cell 1 自动向 GitHub 拉取并同步最新提交
FORCE_RECLONE_REPO = False  # 若想彻底删除 /content/SwellEx 重新克隆，设为 True 运行一次即可
# ------------------------------------------------------------------

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)


def extract_drive_folder_id(url_or_id: str) -> str:
    text = url_or_id.strip()
    for pattern in (r"/folders/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"):
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{15,}", text):
        return text
    raise ValueError(f"无法从输入解析 Google Drive Folder ID: {url_or_id!r}")


ORIGINDATA_FOLDER_ID = extract_drive_folder_id(ORIGINDATA_DRIVE_URL)
REQUIRED_REL_FILES = (
    "positions/positions_vla.txt",
    "environments/ctds/i9605.prn",
    "events/range/RangeEventS5/SproulToVLA.S5.txt",
    *[f"events/S5/vla_matfiles/S5_VLA_NO_{idx}.mat" for idx in range(1, 22)],
)
OPTIONAL_REL_FILES = (
    "source/Info_S5.txt",
    "source/S5_SourceInfo.mat",
)

# 1. 检查是否误开了 GPU 运行时，防止白白消耗 Colab GPU 额度
has_nvidia = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi", "-L"], capture_output=True, text=True
).returncode == 0
if has_nvidia:
    print("⚠️ [额度提醒] 检测到当前实例挂载了 GPU！模块一仅使用纯 CPU（NumPy/SciPy/h5py），")
    print("   建议切换至 Runtime -> Change runtime type -> CPU 以节省 Colab GPU 计算单元。\n")
else:
    print("✅ 当前为纯 CPU 运行环境（零 GPU 额度消耗）。")

# 2. 先挂载 Google Drive（Colab 环境自动挂载；本地环境使用模拟/本地目录）
DRIVE_MOUNT_ROOT = Path("/content/drive")
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)
    DRIVE_BASE = DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR
else:
    DRIVE_BASE = None  # 待 PROJECT_ROOT 解析后回退到 outputs/drive_staging


def get_git_commit_info(repo_dir: Path) -> tuple[str, str]:
    proc = subprocess.run(
        ["git", "-C", str(repo_dir), "log", "-1", "--format=%h|%s"],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and "|" in proc.stdout:
        short_sha, subject = proc.stdout.strip().split("|", 1)
        return short_sha, subject
    return "unknown", ""


def reload_project_modules() -> None:
    for mod_name in [m for m in list(sys.modules) if m == "scripts" or m.startswith("scripts.")]:
        sys.modules.pop(mod_name, None)
    importlib.invalidate_caches()


def ensure_project_root() -> Path:
    if not IN_COLAB:
        for candidate in (Path.cwd(), Path.cwd().parent):
            if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
                return candidate.resolve()
        raise FileNotFoundError("未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。")

    colab_repo = Path("/content/SwellEx")
    if FORCE_RECLONE_REPO and colab_repo.exists():
        shutil.rmtree(colab_repo)
        print(f"🗑️ 已按 FORCE_RECLONE_REPO=True 清除旧目录: {colab_repo}")

    if (colab_repo / ".git").is_dir() and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        old_sha, _ = get_git_commit_info(colab_repo)
        if AUTO_UPDATE_REPO:
            subprocess.run(["git", "-C", str(colab_repo), "remote", "set-url", "origin", REPO_GIT_URL], capture_output=True)
            fetch_proc = subprocess.run(
                ["git", "-C", str(colab_repo), "fetch", "--depth", "1", "origin", REPO_GIT_BRANCH],
                capture_output=True,
                text=True,
            )
            if fetch_proc.returncode == 0:
                subprocess.run(["git", "-C", str(colab_repo), "reset", "--hard", "FETCH_HEAD"], check=True, capture_output=True)
                subprocess.run(
                    ["git", "-C", str(colab_repo), "clean", "-fd", "-e", "outputs", "-e", "output", "-e", "Origindata"],
                    capture_output=True,
                )
                new_sha, subject = get_git_commit_info(colab_repo)
                if old_sha != new_sha:
                    print(f"🔄 已同步云端仓库最新提交: {old_sha} -> {new_sha} ({subject})")
                else:
                    print(f"✅ 云端仓库已是最新提交: {new_sha} ({subject})")
            else:
                print(f"⚠️ 检查远端更新失败（继续使用现有副本 {old_sha}）: {fetch_proc.stderr.strip()}")
        return colab_repo.resolve()

    if colab_repo.exists():
        shutil.rmtree(colab_repo)
    print(f"📥 正在克隆最新云端仓库 {REPO_GIT_URL} ({REPO_GIT_BRANCH}) -> {colab_repo} ...")
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_GIT_BRANCH, REPO_GIT_URL, str(colab_repo)],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        new_sha, subject = get_git_commit_info(colab_repo)
        print(f"✅ 克隆完成: {new_sha} ({subject})")
        return colab_repo.resolve()

    # 若 GitHub 克隆失败（如仓库设为私有），回退查找当前目录或 Google Drive 中的副本
    fallback_candidates = [Path.cwd(), Path.cwd().parent]
    if DRIVE_MOUNT_ROOT.exists():
        fallback_candidates.extend([
            DRIVE_MOUNT_ROOT / "MyDrive" / "SwellEx",
            DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "SwellEx",
        ])
    for candidate in fallback_candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            print(f"⚠️ GitHub 克隆未成功，已回退使用挂载目录副本: {candidate.resolve()}")
            return candidate.resolve()
    raise FileNotFoundError(
        f"自动克隆仓库失败（可能仓库为私有）: {proc.stderr.strip()}\n"
        "请先将最新 SwellEx 仓库上传至 /content/drive/MyDrive/SwellEx 或使用带权限链接克隆。"
    )


PROJECT_ROOT = ensure_project_root()
reload_project_modules()
REPO_COMMIT, REPO_SUBJECT = get_git_commit_info(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if DRIVE_BASE is None:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))

# Colab 下统一使用实例本地 ext4 高速盘作为计算工作区（规避 Drive FUSE 不支持 symlink 与随机写慢）
WORK_ROOT = Path("/content/swellex_work") if IN_COLAB else PROJECT_ROOT
LOCAL_OUTPUTS = WORK_ROOT / "outputs"
if IN_COLAB:
    LOCAL_OUTPUTS.mkdir(parents=True, exist_ok=True)
DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"


def inspect_origindata_candidate(folder: Path) -> tuple[Path | None, list[str]]:
    """同时支持文件夹本身即 Origindata 内容，或文件夹内含 Origindata/ 子目录。"""
    if not folder.exists():
        return None, list(REQUIRED_REL_FILES)
    partial_missing: list[str] = list(REQUIRED_REL_FILES)
    for cand in (folder, folder / "Origindata"):
        if not cand.is_dir():
            continue
        missing = [rel for rel in REQUIRED_REL_FILES if not (cand / rel).is_file()]
        if not missing:
            return cand, []
        if len(missing) < len(partial_missing):
            partial_missing = missing
    return None, partial_missing


def get_drivefs_id(path: Path) -> str | None:
    if not hasattr(os, "getxattr"):
        return None
    for attr in ("user.drive.id", "drive.id"):
        try:
            raw = os.getxattr(path, attr, follow_symlinks=False)
            return raw.decode("utf-8", errors="ignore").strip()
        except OSError:
            continue
    return None


def locate_origindata_on_mounted_drive(mount_root: Path, target_id: str) -> tuple[Path | None, str]:
    # 1) 优先检查快捷方式按 ID 映射路径及常见命名路径
    direct_candidates = [
        (mount_root / ".shortcut-targets-by-id" / target_id, "drive_shortcut_id"),
        (mount_root / "MyDrive" / "Origindata", "mydrive_named_path"),
        (mount_root / "MyDrive" / "SwellEx" / "Origindata", "mydrive_named_path"),
        (mount_root / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "Origindata", "mydrive_named_path"),
    ]
    fallback_match: tuple[Path, str] | None = None
    best_partial: tuple[Path, list[str]] | None = None

    for cand_path, source_tag in direct_candidates:
        if not cand_path.exists():
            continue
        valid_dir, missing = inspect_origindata_candidate(cand_path)
        if valid_dir is not None:
            if source_tag == "drive_shortcut_id" or get_drivefs_id(cand_path) == target_id or get_drivefs_id(valid_dir) == target_id:
                return valid_dir, f"{source_tag}:exact_id"
            if fallback_match is None:
                fallback_match = (valid_dir, source_tag)
        elif len(missing) < len(REQUIRED_REL_FILES):
            best_partial = (cand_path, missing)

    # 2) 浅层广度优先扫描 MyDrive / Shareddrives，精确比对 Folder ID (1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj)
    skip_names = {
        ".Trash", ".ipynb_checkpoints", "__pycache__", ".git", "node_modules",
        "datasets", "experiments", "outputs", "output",
        "environments", "events", "positions", "source", "ctds", "vla_matfiles",
    }
    search_roots = [p for p in (mount_root / "MyDrive", mount_root / "Shareddrives") if p.is_dir()]
    queue: deque[tuple[Path, int]] = deque((root, 0) for root in search_roots)

    while queue:
        curr, depth = queue.popleft()
        if get_drivefs_id(curr) == target_id:
            valid_dir, missing = inspect_origindata_candidate(curr)
            if valid_dir is not None:
                return valid_dir, "drivefs_xattr_exact_id"
            best_partial = (curr, missing)
        elif fallback_match is None:
            if (curr / "positions" / "positions_vla.txt").is_file():
                valid_dir, missing = inspect_origindata_candidate(curr)
                if valid_dir is not None:
                    fallback_match = (valid_dir, "drivefs_structure_match")
                elif len(missing) < len(REQUIRED_REL_FILES):
                    best_partial = (curr, missing)

        if depth >= 5:
            continue
        try:
            with os.scandir(curr) as it:
                for entry in it:
                    if entry.name.startswith(".") or entry.name in skip_names:
                        continue
                    try:
                        if entry.is_dir(follow_symlinks=False):
                            queue.append((Path(entry.path), depth + 1))
                    except OSError:
                        continue
        except OSError:
            continue

    if fallback_match is not None:
        return fallback_match
    if best_partial is not None:
        folder_found, missing_list = best_partial
        preview = ", ".join(missing_list[:5])
        print(f"⚠️ 在 Google Drive 发现疑似目录 {folder_found}，但缺少 {len(missing_list)} 个必需文件（如: {preview}），请检查云盘同步是否完成。")
    return None, "not_found_on_mount"


def download_origindata_via_drive_api(target_id: str, dst_origindata: Path) -> tuple[Path | None, str]:
    """当文件夹为外部共享且未创建 MyDrive 快捷方式时，直接通过 Drive API v3 按 Folder ID 拉取所需文件。"""
    from google.colab import auth  # type: ignore
    from googleapiclient.discovery import build  # type: ignore
    from googleapiclient.http import MediaIoBaseDownload  # type: ignore

    print(f"🌐 正在通过 Google Drive API v3 解析文件夹 ID: {target_id} ...")
    auth.authenticate_user()
    service = build("drive", "v3", cache_discovery=False)

    # 若挂载盘存在但目录层级较深，先通过 API 还原 MyDrive 完整父路径直接走本地挂载复制
    if DRIVE_MOUNT_ROOT.is_dir():
        chain: list[str] = []
        curr_id: str | None = target_id
        seen: set[str] = set()
        while curr_id and curr_id not in seen:
            seen.add(curr_id)
            info = service.files().get(
                fileId=curr_id, fields="id, name, parents", supportsAllDrives=True
            ).execute()
            parents = info.get("parents") or []
            if not parents:
                break
            chain.append(info["name"])
            curr_id = parents[0]
        if chain:
            mounted_cand = DRIVE_MOUNT_ROOT / "MyDrive" / Path(*reversed(chain))
            valid_cand, _ = inspect_origindata_candidate(mounted_cand)
            if valid_cand is not None:
                stage_origindata_to_local_ssd(valid_cand, dst_origindata)
                return dst_origindata, f"drive_api_parent_path:{valid_cand}"

    def list_children(folder_id: str) -> dict[str, dict]:
        items: dict[str, dict] = {}
        page_token = None
        while True:
            resp = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageSize=1000,
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
            for item in resp.get("files", []):
                items[item["name"]] = item
            page_token = resp.get("nextPageToken")
            if not page_token:
                break
        return items

    root_children = list_children(target_id)
    if "Origindata" in root_children and root_children["Origindata"]["mimeType"] == "application/vnd.google-apps.folder":
        root_children = list_children(root_children["Origindata"]["id"])

    folder_cache: dict[tuple[str, ...], dict[str, dict]] = {(): root_children}

    def resolve_remote_file(rel_posix: str) -> dict | None:
        parts = tuple(rel_posix.split("/"))
        curr_children = root_children
        for idx, dir_name in enumerate(parts[:-1]):
            prefix = parts[: idx + 1]
            if prefix not in folder_cache:
                dir_meta = curr_children.get(dir_name)
                if not dir_meta or dir_meta["mimeType"] != "application/vnd.google-apps.folder":
                    return None
                folder_cache[prefix] = list_children(dir_meta["id"])
            curr_children = folder_cache[prefix]
        return curr_children.get(parts[-1])

    for rel in (*REQUIRED_REL_FILES, *OPTIONAL_REL_FILES):
        meta = resolve_remote_file(rel)
        if meta is None:
            if rel in REQUIRED_REL_FILES:
                raise FileNotFoundError(f"Google Drive 文件夹 ({target_id}) 中缺少必需文件: {rel}")
            continue
        dst_file = dst_origindata / rel
        remote_size = int(meta.get("size", -1))
        if dst_file.is_file() and (remote_size < 0 or dst_file.stat().st_size == remote_size):
            continue
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        request = service.files().get_media(fileId=meta["id"], supportsAllDrives=True)
        tmp_file = dst_file.with_suffix(dst_file.suffix + ".part")
        with tmp_file.open("wb") as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        tmp_file.replace(dst_file)
    return dst_origindata, f"drive_api_v3:{target_id}"


def stage_origindata_to_local_ssd(src_origindata: Path, dst_origindata: Path) -> Path:
    """将计算所需的 24 个核心原始文件复制到 Colab 本地 SSD，避免多轮 FUSE 读取与 mtime 抖动。"""
    if src_origindata.resolve() == dst_origindata.resolve():
        return dst_origindata
    copied = 0
    for rel in (*REQUIRED_REL_FILES, *OPTIONAL_REL_FILES):
        src_file = src_origindata / rel
        if not src_file.is_file():
            continue
        dst_file = dst_origindata / rel
        if dst_file.is_file() and dst_file.stat().st_size == src_file.stat().st_size:
            continue
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_file, dst_file)
        copied += 1
    if copied:
        print(f"⚡ 已将 {copied} 个原始数据文件从 Google Drive 缓存至本地高速盘: {dst_origindata}")
    return dst_origindata


# 3. 解析并准备 WORK_ROOT / "Origindata"
work_origindata = WORK_ROOT / "Origindata"
ORIGINDATA_RESOLUTION = "unresolved"
ORIGINDATA_SOURCE_PATH = ""

existing_local, _ = inspect_origindata_candidate(work_origindata)
if existing_local is not None and existing_local == work_origindata:
    ORIGINDATA_RESOLUTION = "local_ready"
    ORIGINDATA_SOURCE_PATH = str(work_origindata)
elif ORIGINDATA_LOCAL_OVERRIDE:
    override_valid, missing = inspect_origindata_candidate(Path(ORIGINDATA_LOCAL_OVERRIDE))
    if override_valid is None:
        raise FileNotFoundError(
            f"ORIGINDATA_LOCAL_OVERRIDE={ORIGINDATA_LOCAL_OVERRIDE!r} 缺少必需文件（共缺 {len(missing)} 个，如 {missing[:3]}）。"
        )
    ORIGINDATA_SOURCE_PATH = str(override_valid)
    if IN_COLAB:
        stage_origindata_to_local_ssd(override_valid, work_origindata)
    elif not work_origindata.exists():
        work_origindata.symlink_to(override_valid, target_is_directory=True)
    ORIGINDATA_RESOLUTION = "local_override"
elif IN_COLAB:
    mounted_src, mode = locate_origindata_on_mounted_drive(DRIVE_MOUNT_ROOT, ORIGINDATA_FOLDER_ID)
    if mounted_src is not None:
        ORIGINDATA_SOURCE_PATH = str(mounted_src)
        stage_origindata_to_local_ssd(mounted_src, work_origindata)
        ORIGINDATA_RESOLUTION = mode
    else:
        api_dst, mode = download_origindata_via_drive_api(ORIGINDATA_FOLDER_ID, work_origindata)
        ORIGINDATA_SOURCE_PATH = f"gdrive://{ORIGINDATA_FOLDER_ID}"
        ORIGINDATA_RESOLUTION = mode
else:
    repo_orig, _ = inspect_origindata_candidate(PROJECT_ROOT / "Origindata")
    if repo_orig is not None:
        ORIGINDATA_RESOLUTION = "local_repo"
        ORIGINDATA_SOURCE_PATH = str(repo_orig)

ORIGINDATA_READY = inspect_origindata_candidate(work_origindata)[0] == work_origindata

# 4. 检查数值依赖（Colab 默认自带 numpy / scipy / h5py）
REQUIRED_PKGS = ("numpy", "scipy", "h5py")
MISSING_PKGS = [pkg for pkg in REQUIRED_PKGS if importlib.util.find_spec(pkg) is None]

print(f"PROJECT_ROOT       : {PROJECT_ROOT} (commit={REPO_COMMIT})")
print(f"WORK_ROOT (本地盘) : {WORK_ROOT}")
print(f"Drive Folder ID    : {ORIGINDATA_FOLDER_ID}")
print(f"Origindata 就绪    : {ORIGINDATA_READY} (来源={ORIGINDATA_RESOLUTION}, 路径={ORIGINDATA_SOURCE_PATH or work_origindata})")
print(f"DRIVE_DATASET_ROOT : {DRIVE_DATASET_ROOT}")
if MISSING_PKGS:
    print(f"ℹ️ 当前环境缺少数值包 {MISSING_PKGS}（后续单元将以配置预览/干跑模式执行；在 Colab 可运行 `!pip install -q h5py numpy scipy`）。")
else:
    print("✅ 数值依赖 (numpy, scipy, h5py) 已全部就绪。")

## 1. 选择要生成的实验预设与特征包

下方参数单元直接复用仓库内置的官方 TOML 配置（无硬编码魔法参数）：
- `PRESET`：
  - `"train_before"`：主实验预设（`range_overlap`，最接近点前训练、后测试，训练侧 800 窗（再按 3:1 划分训练/验证），测试 700 窗）
  - `"train_after"`：反向泛化预设（`range_overlap`，最接近点后训练、前测试）
  - `"full"`：整段交错划分预设（`full_interleaved`，4:1 训练/测试）
- `QUICK_SMOKE_END_S`：设为 `None` 生成完整 75 分钟 S5 数据；若首次联调想在几十秒内验证端到端流程，可设为 `299`（仅截取前 300 秒并自动切换到 `full_interleaved`）。
- `FORCE_REGENERATE`：默认 `False`，同一会话内已通过校验的RBD / SCM 单路 HDF5 会直接复用，中断残留目录会自动清理重建。

In [ ]:
# Cell 2: 生成任务参数配置（按需修改）
from copy import deepcopy
import tomllib

# --- 用户可调配置 ---
PRESET = "train_before"            # 可选: "train_before" | "train_after" | "full"
QUICK_SMOKE_END_S = None           # 完整生成保持 None；快速端到端冒烟可设为 299

BUILD_BASE_ROUTES = True           # 生成 RBD / SCM 单路 HDF5（供 base_pack 使用）
FORCE_REGENERATE = False           # True=强制删除本地已生成的同配置 HDF5 重新计算；False=复用已完好通过的结果
# --------------------

PRESET_TOML_MAP = {
    "train_before": PROJECT_ROOT / "scripts/remote_workflow/presets/train_before.toml",
    "train_after": PROJECT_ROOT / "scripts/remote_workflow/presets/train_after.toml",
    "full": PROJECT_ROOT / "scripts/remote_workflow/presets/full.toml",
}

if PRESET not in PRESET_TOML_MAP:
    raise ValueError(f"未知 PRESET={PRESET!r}，可选: {list(PRESET_TOML_MAP)}")

raw_preset_cfg = tomllib.loads(PRESET_TOML_MAP[PRESET].read_text(encoding="utf-8"))
dataset_cfg = deepcopy(raw_preset_cfg["dataset"])
rbd_cfg = deepcopy(raw_preset_cfg["rbd"])
dataset_tag = PRESET

if QUICK_SMOKE_END_S is not None:
    dataset_cfg["segment_end_s"] = int(QUICK_SMOKE_END_S)
    dataset_cfg["split_strategy"] = "full_interleaved"
    dataset_cfg["manual_dataset_variant_tag"] = f"colab_smoke_{int(QUICK_SMOKE_END_S)}s"
    dataset_tag = f"{PRESET}_smoke_{int(QUICK_SMOKE_END_S)}s"

GENERATION_PAYLOAD = {
    "schema_version": 1,
    "dataset": dataset_cfg,
    "rbd": rbd_cfg,
    "result_path": f"outputs/generation_report_{dataset_tag}.json",
}

print(f"已加载预设配置: {PRESET_TOML_MAP[PRESET].relative_to(PROJECT_ROOT)}")
print(f"  dataset_tag    : {dataset_tag}")
print(f"  split_strategy : {dataset_cfg['split_strategy']} (train_side={dataset_cfg['train_side']})")
print(f"  时间范围       : {dataset_cfg['segment_start_s']}s .. {dataset_cfg['segment_end_s']}s (步长 {dataset_cfg['segment_step_s']}s)")


## 2. 生成标准单路特征（`RBD` / `SCM`）并执行契约校验

所有写入均在实例本地高速盘 `<WORK_ROOT>/outputs/Datasets/` 完成，并自动调用 `inspect_h5`、`split_check` 与 `crossroute_check` 验证样本数、频点轴、阵元对一维列主序索引及两路跨方法对齐。

In [ ]:
# Cell 3: 提取单路 RBD / SCM 特征并做 HDF5 结构契约校验
import json

GENERATED_LOCAL: dict[str, dict] = {}


def load_or_generate_routes(
    norm_cfg: dict,
    *, 
    generate_fn,
    ident_hash: str,
    work_root: Path,
    force: bool,
) -> dict:
    report_file = Path(norm_cfg["result_path"])
    if not force and report_file.is_file():
        cached = json.loads(report_file.read_text(encoding="utf-8"))
        dirs_ok = cached.get("status") == "PASS" and all(
            Path(p).is_dir() and any(Path(p).glob("*_train.h5")) and any(Path(p).glob("*_test.h5"))
            for p in cached.get("datasets", {}).values()
        )
        if dirs_ok:
            print(f"♻️ 复用本地已完成的特征目录 (hash={ident_hash})")
            return cached
    datasets_parent = work_root / "outputs" / "Datasets"
    if datasets_parent.is_dir():
        for stale in datasets_parent.glob(f"*_{ident_hash}"):
            if stale.is_dir():
                shutil.rmtree(stale)
    return generate_fn(norm_cfg, root=work_root)


if MISSING_PKGS or not ORIGINDATA_READY:
    print("⏭️ 跳过实际特征计算（缺少数值运行库或 Origindata 未就绪）。配置结构校验通过。")
elif BUILD_BASE_ROUTES:
    from scripts.data_processing.generation_config import normalize_config
    from scripts.data_processing.generation_core import generate_datasets, generation_identity
    from scripts.data_processing.dataset_validation import (
        crossroute_check,
        inspect_h5,
        split_check,
    )
    from scripts.data_processing.reporting import normalize_expected

    norm_cfg = normalize_config(GENERATION_PAYLOAD, root=WORK_ROOT)
    ident = generation_identity(norm_cfg)
    print(f"🚀 准备单路 RBD / SCM HDF5 ({dataset_tag}, hash={ident}) ...")
    gen_result = load_or_generate_routes(
        norm_cfg,
        generate_fn=generate_datasets,
        ident_hash=ident,
        work_root=WORK_ROOT,
        force=FORCE_REGENERATE,
    )
    route_dirs = {method: Path(path) for method, path in gen_result["datasets"].items()}

    # 对单路两组 HDF5 做单文件、切分不重叠与跨路由一致性契约校验
    expected_cfg = normalize_expected(norm_cfg)
    all_records = []
    for method, folder in route_dirs.items():
        method_records = []
        for h5_path in sorted(p for p in folder.glob("*.h5") if not p.name.startswith("._")):
            rec, _ = inspect_h5(h5_path, method, expected_cfg)
            method_records.append(rec)
            all_records.append(rec)
        split_check(method_records, expected_cfg)
    crossroute_check(all_records)

    rbd_meta = gen_result["dataset_results"]["RBD"]
    GENERATED_LOCAL["base_routes"] = {
        "dataset_tag": dataset_tag,
        "routes": {k: str(v) for k, v in route_dirs.items()},
        "train_count": rbd_meta["train_count"],
        "test_count": rbd_meta["test_count"],
        "frequency_hz": rbd_meta["frequency_hz"],
    }
    print("✅ 单路 HDF5 生成与两路契约验证通过:")
    for method, folder in route_dirs.items():
        print(f"   - {method:4s}: {folder.relative_to(WORK_ROOT)}")
    print(
        f"   样本统计: train={rbd_meta['train_count']}, test={rbd_meta['test_count']}, "
        f"频点数={len(rbd_meta['frequency_hz'])}"
    )

## 3. 同步 HDF5 到 Google Drive 并写入 `dataset_manifest.json`

将本地 SSD 生成完毕的数据集整目录复制到 `DRIVE_DATASET_ROOT`（`/content/drive/MyDrive/SwellEx_Colab/datasets/`），并写入包含原始数据溯源（Folder ID `1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj`）的 `dataset_manifest.json`。完成后即可**关闭并释放当前 CPU 运行时**，切换到 **模块二（GPU）**。

In [ ]:
# Cell 4: 将生成好的 HDF5 数据集复制到 Google Drive 并更新 dataset_manifest.json
from datetime import datetime, timezone
import hashlib
import json

MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def sync_dir_to_drive(src_dir: Path, rel_dest: Path) -> dict:
    dst_dir = DRIVE_DATASET_ROOT / rel_dest
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src_dir, dst_dir)
    h5_files = {}
    for h5_file in sorted(dst_dir.glob("*.h5")):
        h5_files[h5_file.name] = {
            "size_mb": round(h5_file.stat().st_size / (1024 * 1024), 3),
            "sha256": sha256_file(h5_file),
        }
    return {
        "rel_dir": rel_dest.as_posix(),
        "files": h5_files,
    }


if not GENERATED_LOCAL:
    print("ℹ️ 本次未产生新的本地 HDF5 目录，展示 Google Drive 目标结构示例:")
    example_layout = {
        "origindata_drive_folder_id": ORIGINDATA_FOLDER_ID,
        "origindata_drive_url": ORIGINDATA_DRIVE_URL,
        "drive_dataset_root": str(DRIVE_DATASET_ROOT),
        "manifest_file": str(MANIFEST_PATH),
        "entries": {
            f"presets/{dataset_tag}/base_routes/<RBD|SCM>": "*_train.h5, *_test.h5",
        },
    }
    print(json.dumps(example_layout, ensure_ascii=False, indent=2))
else:
    DRIVE_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    manifest = (
        json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if MANIFEST_PATH.exists()
        else {"schema_version": 1, "presets": {}}
    )
    manifest["origindata_source"] = {
        "drive_folder_id": ORIGINDATA_FOLDER_ID,
        "drive_url": ORIGINDATA_DRIVE_URL,
        "resolution_mode": ORIGINDATA_RESOLUTION,
        "source_path": ORIGINDATA_SOURCE_PATH,
    }
    manifest["repo_commit"] = REPO_COMMIT
    preset_entry = {
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "repo_commit": REPO_COMMIT,
    }
    manifest["presets"][dataset_tag] = preset_entry

    if "base_routes" in GENERATED_LOCAL:
        preset_entry["base_routes"] = {}
        for method, src_str in GENERATED_LOCAL["base_routes"]["routes"].items():
            src = Path(src_str)
            preset_entry["base_routes"][method] = sync_dir_to_drive(
                src, Path("presets") / dataset_tag / "base_routes" / method
            )

    MANIFEST_PATH.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(f"🎉 已全部同步至 Google Drive 并更新清单: {MANIFEST_PATH}")
    print("👉 现在可以断开当前 CPU 运行时，打开 notebooks/02_network_training_gpu.ipynb 连接 GPU 开始训练！")